In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [3]:
urls = {
    "Charles Oliveira": "http://ufcstats.com/fighter-details/07225ba28ae309b6",
    "Max Holloway": "http://ufcstats.com/fighter-details/150ff4cc642270b9"
}

In [4]:
def get_fighter_html(url):
    headers = {
         'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        return BeautifulSoup(response.content, 'html.parser')
    else:
        print(f"deu erro {response.status_code}")
        return None

In [6]:
charles = get_fighter_html(urls["Charles Oliveira"])
print("HTML do Charles baixado com sucesso! Tamanho:", len(str(charles)))

HTML do Charles baixado com sucesso! Tamanho: 109424


## Caracteristicas dos lutadores

In [7]:
def extract_fighter_bio(soup, name):
    fighter_data = {"name": name}

    info_box = soup.find('div', class_='b-list__info-box_style_small-width')

    if info_box:
        items = info_box.find_all('li')
        for item in items:
            text_line = item.text.strip().split('\n')
            if len(text_line) >= 2:
                key = text_line[0].strip().replace(':', '')
                value = text_line[-1].strip()
                fighter_data[key] = value
    
    return fighter_data

In [8]:
charles_bio = extract_fighter_bio(charles, "Charles Oliveira")
print("dados do Charles:")
for k,v in charles_bio.items():
    print(f"{k}: {v}")

dados do Charles:
name: Charles Oliveira
Height: 5' 10"
Weight: 155 lbs.
Reach: 74"
STANCE: Orthodox
DOB: Oct 17, 1989


### Puxando estatisticas completas 

In [15]:
def extract_fighter_stats(soup, name):
    fighter_data = {"name": name}

    info_lists = soup.find_all('ul', class_='b-list__box-list')

    if not info_lists:
        return fighter_data

    for info_list in info_lists:
        items = info_list.find_all('li')
        for item in items:
            text_line = [line.strip() for line in item.text.strip().split('\n') if line.strip()]

            if len(text_line) >= 2:
                key = text_line[0].replace(':', '')
                value = text_line[-1]

                if key and value: 
                    fighter_data[key] = value
    
    return fighter_data

In [16]:
for name, url in urls.items():
    soup = get_fighter_html(url)
    stats = extract_fighter_stats(soup, name)
    print(f"\n Estatísticas de {name}")

    for k, v in stats.items():
        print(f"{k}: {v}")


 Estatísticas de Charles Oliveira
name: Charles Oliveira
Height: 5' 10"
Weight: 155 lbs.
Reach: 74"
STANCE: Orthodox
DOB: Oct 17, 1989
SLpM: 3.35
Str. Acc.: 54%
SApM: 3.24
Str. Def: 49%
TD Avg.: 2.22
TD Acc.: 40%
TD Def.: 54%
Sub. Avg.: 2.6

 Estatísticas de Max Holloway
name: Max Holloway
Height: 5' 11"
Weight: 155 lbs.
Reach: 69"
STANCE: Orthodox
DOB: Dec 04, 1991
SLpM: 7.20
Str. Acc.: 48%
SApM: 4.74
Str. Def: 59%
TD Avg.: 0.24
TD Acc.: 53%
TD Def.: 83%
Sub. Avg.: 0.3


### Puxando histórico de lutas

In [22]:
def extract_fight_history(soup, fighter_name):
    history = []
    
    table = soup.find('table', class_='b-fight-details__table')
    if not table:
        return history
    
    rows = table.find_all('tr')[1:]
    
    for row in rows:
        cols = row.find_all('td')
        if len(cols) >= 10:
            result = cols[0].find('a').text.strip() if cols[0].find('a') else "N/A"
            
            fighters_tags = cols[1].find_all('a')
            if len(fighters_tags) == 2:
                name1 = ' '.join(fighters_tags[0].text.split())
                name2 = ' '.join(fighters_tags[1].text.split())
                opponent = name1 if name1 != fighter_name else name2
            else:
                opponent = "desconhecido"
                
            method = cols[7].text.strip().split('\n')[0]
            rnd = cols[8].text.strip()
            time = cols[9].text.strip()
        
            fight_data = {
                "Fighter": fighter_name,
                "Opponent": opponent,
                "Result": result,
                "Method": method,
                "Round": rnd,
                "Time": time
            }
            history.append(fight_data)
            
    return history



In [23]:
soup_max = get_fighter_html(urls["Max Holloway"])
history_max = extract_fight_history(soup_max, "Max Holloway")

for fight in history_max[:3]:
    print(fight)

{'Fighter': 'Max Holloway', 'Opponent': 'Dustin Poirier', 'Result': 'win', 'Method': 'U-DEC', 'Round': '5', 'Time': '5:00'}
{'Fighter': 'Max Holloway', 'Opponent': 'Ilia Topuria', 'Result': 'loss', 'Method': 'KO/TKO', 'Round': '3', 'Time': '1:34'}
{'Fighter': 'Max Holloway', 'Opponent': 'Justin Gaethje', 'Result': 'win', 'Method': 'KO/TKO', 'Round': '5', 'Time': '4:59'}


### Salvando o parquet

In [25]:
all_stats = []
all_fights = []

for name, url in urls.items():
    soup = get_fighter_html(url)

    if soup:
        fighter_stats = extract_fighter_stats(soup, name)
        all_stats.append(fighter_stats)
        
        fighter_history = extract_fight_history(soup, name)
        all_fights.extend(fighter_history) 

df_stats = pd.DataFrame(all_stats)
df_fights = pd.DataFrame(all_fights)

display(df_stats.head())

import os
os.makedirs("../data/raw", exist_ok=True)
df_stats.to_parquet("../data/raw/fighter_stats.parquet", index=False)
df_fights.to_parquet("../data/raw/fight_history.parquet", index=False)

,name,Height,Weight,Reach,STANCE,DOB,SLpM,Str. Acc.,SApM,Str. Def,TD Avg.,TD Acc.,TD Def.,Sub. Avg.
0,Charles Oliveira,"5' 10""",155 lbs.,"74""",Orthodox,"Oct 17, 1989",3.35,54%,3.24,49%,2.22,40%,54%,2.6
1,Max Holloway,"5' 11""",155 lbs.,"69""",Orthodox,"Dec 04, 1991",7.20,48%,4.74,59%,0.24,53%,83%,0.3
